In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# List contents of the Info vis directory
info_vis_path = '/content/drive/My Drive/Info vis/'
print(os.listdir(info_vis_path))


In [ ]:
import os
import pandas as pd

In [ ]:
import librosa
import numpy as np


In [ ]:
BASE_DIR = '/content/drive/My Drive/Info vis/ALL BIRDS/'

In [ ]:
BASE_DIR = '/content/drive/My Drive/Info vis/AllBirdsv4.csv'


In [ ]:
df_metadata = pd.read_csv(os.path.join(os.path.dirname(BASE_DIR), 'AllBirdsv4.csv'))
df_metadata['File ID'] = df_metadata['File ID'].astype(str)

In [ ]:
df_metadata = pd.read_csv(os.path.join(os.path.dirname(BASE_DIR), 'AllBirdsv4.csv'))
df_metadata['File ID'] = df_metadata['File ID'].astype(str)

Extract MFCC features from MP3 files

In [ ]:
import os
import pandas as pd
import numpy as np
import librosa
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split

CSV_DIR = '/content/drive/My Drive/Info vis/'
AUDIO_DIR = os.path.join(CSV_DIR, 'ALL BIRDS')

print("---Data Loading and Filtering---")
try:

    #Load training data(AllBirdsv4.csv)
    csv_path_train = os.path.join(CSV_DIR, 'AllBirdsv4.csv')
    df_train = pd.read_csv(csv_path_train)
    df_train['File ID'] = df_train['File ID'].astype(str) #convert to str to match the .mp3 file name easily


    #Filter audio by Quality C or better
    quality_filter = ['A', 'B', 'C']
    df_features = df_train[df_train['Quality'].isin(quality_filter)].copy()


    print(f"CSV loaded successfully. Training data count: {len(df_features)}")
    if len(df_features) == 0:
      print(" Zero data records after filtering.")
      exit()

except FileNotFoundError:
    print(f"File not found: {csv_path_train}")
    exit()


# MFCC

def extract_mfcc(file_path, duration=5.0, n_mfcc=13):
#Extract in first 5 second to match all the lenth. n_mfcc = 13 is for MFCC extract
  try:
     # Load audio file (Warning may appear but loading proceeds if the path is correct)
     audio, sr = librosa.load(file_path, sr=22050, duration=duration)
     # sr = 22050 standard sampling rate for bioacoustic analysis

     mfccs = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)

    # Average across the time
     mfccs_processed = np.mean(mfccs.T, axis=0)
     return mfccs_processed

  except Exception:
    return None

# Execute feature extraction

feature_list = []
# extracted mfcc feature will be saved in feature_list[]

print("\n---Training Data Feature Extraction---")

for index, row in tqdm(df_features.iterrows(), total=len(df_features)):
  species_name = row['English_name']
  file_id = row['File ID']


  species_hyphenated = species_name.replace(' ', '-')
  file_name = f"{species_hyphenated}-{file_id}.mp3"

  file_path = os.path.join(AUDIO_DIR, file_name)

  features = extract_mfcc(file_path)

  if features is not None:
    feature_list.append({
        'features': features,
        'label': species_name
    })

    df_extracted_features = pd.DataFrame(feature_list)

    print("\n---Final Extraction Result---")
    print(f"Successful feature extraction for {len(df_extracted_features)} / {len(df_features)} records")


# for next step: Model training

if len(df_extracted_features) > 0:
  print("\n Feature extraction successful! Use this DataFrame to train the model.")
  x = np.array(df_extracted_features['features'].tolist())
  y = np.array(df_extracted_features['label'].tolist())

else:
    print("Extraction failed.")




In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# --- 1. Encode labels ---
le = LabelEncoder()
y_encoded = le.fit_transform(y)
class_names = le.classes_

print(f"Number of bird species for training: {len(class_names)}")
print("Class order used in confusion matrix:")
for i, name in enumerate(class_names):
    print(f"{i:2d}: {name}")

# --- 2. Train / validate split (same as before) ---
X_train, X_val, y_train, y_val = train_test_split(
    x, y_encoded, test_size=0.2, random_state=42
)

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# --- 3. Confusion matrix ---
y_pred = model.predict(X_val)
cm = confusion_matrix(y_val, y_pred)   # rows = true, cols = predicted

print("\nConfusion matrix shape:", cm.shape)

# --- 4. Pretty heatmap (same as before) ---
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm,
    annot=False,
    fmt='d',
    cmap='Blues',
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel('Predicted Species')
plt.ylabel('True Species')
plt.title('Confusion Matrix')
plt.show()

# Optional: show as a labeled DataFrame for sanity-checking
df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)
display(df_cm)

print("\nModel training completed. The model is now ready to classify the 15 Kasios files.")

# --- 5. OUTPUT FOR D3 / JAVASCRIPT ---------------------

# a) Python list of labels (for reference / copy)
print("\nPython labels list:")
print(class_names.tolist())

# b) Python list-of-lists matrix (for reference)
print("\nPython matrix (list of lists):")
py_matrix = cm.tolist()
for row in py_matrix:
    print(row)

# c) JS-ready code you can paste into your D3 HTML
print("\n\n=== JAVASCRIPT / D3 SNIPPET ===")
print("var labels = [")
for i, name in enumerate(class_names):
    comma = "," if i < len(class_names) - 1 else ""
    print(f"  \"{name}\"{comma}")
print("];\n")

print("var matrix = [")
for i, row in enumerate(py_matrix):
    row_str = ", ".join(str(int(v)) for v in row)
    comma = "," if i < len(py_matrix) - 1 else ""
    print(f"  [{row_str}]{comma}")
print("];")
print("=== END JAVASCRIPT SNIPPET ===")


In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_val, y_pred)
print(f"Overall Accuracy: {accuracy:.2f}")

from sklearn.metrics import precision_score, recall_score

precision = precision_score(y_val, y_pred, average=None)
recall = recall_score(y_val, y_pred, average=None)

for cls, p, r in zip(class_names, precision, recall):
    print(f"{cls}: Precision={p:.2f}, Recall={r:.2f}")

import numpy as np

x = np.arange(len(class_names))  # positions for bars
width = 0.35

plt.figure(figsize=(12, 6))
plt.bar(x - width/2, precision, width, label='Precision', color='skyblue')
plt.bar(x + width/2, recall, width, label='Recall', color='orange')

plt.xticks(x, class_names, rotation=90)
plt.ylabel('Score')
plt.title('Precision and Recall per Bird Species')
plt.legend()
plt.tight_layout()
plt.show()


The darker the cell, the more accurate the prediction for that bird call.



In [ ]:
import os
import matplotlib.pyplot as plt
import pandas as pd

CSV_DIR = '/content/drive/My Drive/Info vis/'

# File paths
ALL_BIRDS_CSV = os.path.join(CSV_DIR, 'AllBirdsv4.csv')
KASIOS_CSV = os.path.join(CSV_DIR, 'Test Birds Location.csv')

df_all = pd.read_csv(ALL_BIRDS_CSV)
df_all['X'] = pd.to_numeric(df_all['X'], errors='coerce')
df_all['Y'] = pd.to_numeric(df_all['Y'], errors='coerce')
df_kasios = pd.read_csv(KASIOS_CSV)

# Separate Rose-crested Blue Pipit vs other species
df_pipit = df_all[df_all['English_name'] == 'Rose-crested Blue Pipit'].copy()
df_others = df_all[df_all['English_name'] != 'Rose-crested Blue Pipit'].copy()

df_kasios = pd.read_csv(KASIOS_CSV)
df_kasios.columns = df_kasios.columns.str.strip()  # remove extra spaces

# Convert coordinate columns to numeric
df_kasios['X'] = pd.to_numeric(df_kasios['X'], errors='coerce')
df_kasios['Y'] = pd.to_numeric(df_kasios['Y'], errors='coerce')

MAP_FILE_NAME = 'Lekagul Roadways 2018.bmp'
map_image_path = os.path.join(CSV_DIR, MAP_FILE_NAME)

map_img = None
try:
    # Attempt to load the BMP image file
    map_img = plt.imread(map_image_path)
    print(f" Background map '{MAP_FILE_NAME}' loaded successfully.")
except FileNotFoundError:
    print(f"Error: Background map '{MAP_FILE_NAME}' not found at path '{map_image_path}'.")
except Exception as e:
    print(f"Warning: An error occurred while loading the map file: {e}")


plt.figure(figsize=(10, 10))

xmin = min(df_all['X'].min(), df_kasios['X'].min())
xmax = max(df_all['X'].max(), df_kasios['X'].max())
ymin = min(df_all['Y'].min(), df_kasios['Y'].min())
ymax = max(df_all['Y'].max(), df_kasios['Y'].max())

# 2. Background map scaled to data coordinates
if map_img is not None:
    plt.imshow(map_img, extent=[xmin, xmax, ymin, ymax], aspect='auto', alpha=0.8)

# 3. Other birds (gray)
plt.scatter(df_others['X'].sample(n=500, random_state=42),
            df_others['Y'].sample(n=500, random_state=42),
            s=5, alpha=0.1, color='gray', label='Other Bird Sightings (Historical)')

# 4. Pipit hotspots (red)
plt.scatter(df_pipit['X'], df_pipit['Y'],
            s=20, alpha=0.8, color='red', label='Historical Pipit Recordings (Hotspots)')

# 5. Kasios test sites (blue stars)
plt.scatter(df_kasios['X'], df_kasios['Y'],
            s=100, marker='*', alpha=1.0, color='blue', label='Kasios Test Locations (15 Records)')

# 6. Labels and layout
plt.title('Comparison of Historical Pipit Habitat vs. Kasios Sampling Sites\n(Scaled to Data Coordinates)')
plt.xlabel('X Coordinate')
plt.ylabel('Y Coordinate')
plt.xlim(xmin, xmax)
plt.ylim(ymin, ymax)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.gca().set_aspect('equal', adjustable='box')
plt.show()

In [ ]:
# This is for machine learning modeling. Will feed and teach ML language to identify the bird sounds



In [ ]:
df_train = pd.read_csv(CSV_PATH)
df_train['File ID'] = df_train['File ID'].astype(str)
quality_filter = ['A', 'B', 'C']
df_features = df_train[df_train['Quality'].isin(quality_filter)].copy()

# Match filename
def find_matching_file(species, file_id):
    target_fragment = f"{species.replace(' ', '-')}-{file_id}".lower()
    for f in os.listdir(AUDIO_DIR):
        if target_fragment in f.lower() and f.lower().endswith('.mp3'):
            return os.path.join(AUDIO_DIR, f)
    return None

species_spectra = {}
for species in df_features['English_name'].unique():
    sample = df_features[df_features['English_name'] == species].iloc[0]
    file_id = sample['File ID']
    file_path = find_matching_file(species, file_id)
    if not file_path:
        print(f"File not found for {species} ({file_id})")
        continue

    try:
        audio, sr = librosa.load(file_path, sr=22050, duration=5.0)
        stft = librosa.stft(audio)
        spectrum = abs(stft).mean(axis=1)
        freqs = librosa.fft_frequencies(sr=sr)
        species_spectra[species] = (freqs, spectrum)
    except Exception as e:
        print(f"Error processing {species} ({file_id}): {e}")

# Plot all species
plt.figure(figsize=(14, 10))
for species, (freqs, spectrum) in species_spectra.items():
    plt.plot(freqs, spectrum, label=species, alpha=0.6)

plt.title("Average Frequency Spectrum by Species")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Amplitude")
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), fontsize='small')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# sound wave modeling